# 10 — Post-development holdout evaluation

This notebook evaluates the **already-registered LSTM v1** using three synthetic **post-development** holdouts. They are **not** the train/validation/test split used in model development. The notebook can safely reuse the eight persisted Unity Catalog tables on subsequent runs.

**Safety:** Defaults are `DATA_MODE="reuse"`, `RUN_INFERENCE=False`, and `EXPORT_FIGURES=False`. No training, registry updates, or table overwrites occur. Inference predictions remain in memory unless exported explicitly.

## 1 — Environment and imports

In [0]:
import os
import site
from pathlib import Path

username = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)

project_root = Path(
    f"/Workspace/Users/{username}/finance-timeseries-ml-platform"
)

os.chdir(project_root)

site.addsitedir("src")

# print("Project root:", project_root)

In [0]:
%uv sync --frozen

In [0]:
%restart_python

In [0]:
from pathlib import Path
import sys
import hashlib
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "src" / "finance_ml").is_dir(), project_root
sys.path.insert(0, str(project_root / "src"))

from finance_ml.pipelines.holdout_data_pipeline import (
    DEFAULT_TABLES, PROTECTED_TABLES, prepare_holdout_data,
)
from finance_ml.pipelines.batched_inference_pipeline import (
    run_batched_inference_pipeline,
)
# print("Project root:", project_root)

## 2 — Configuration and run modes

Use `reuse` after initial table creation. Choose `create` **only if all eight destinations are absent**; never rerun table creation for existing tables. Model scoring is separate from data creation.

In [0]:
N_EXISTING_ACCOUNTS = 2000
HISTORICAL_DAYS = 60
FUTURE_DAYS = 60
NEW_ACCOUNTS = 1000
START_DATE = "2026-01-01"
SIMULATION_SEED = 42
TEMPORAL_CORRUPTION_SEED = 124
ACCOUNT_CORRUPTION_SEED = 125
MODEL_VERSION = "1"
HISTORY_SILVER = "finance_ml.silver.account_events_clean"
HISTORY_GOLD = "finance_ml.gold.account_features"
EXPECTED_HISTORY_SHA256 = "8654cb318f6e130d9d1b02f4bea02b2e762e9ebdc07809332af5dac345daa940"
FUTURE_START = pd.Timestamp(START_DATE) + pd.Timedelta(days=HISTORICAL_DAYS)

DATA_MODE = "reuse"  # "reuse" or "create"; create ONLY when all eight destinations are absent
RUN_INFERENCE = False  # Explicit opt-in to full registered-model scoring
ACCOUNTS_PER_BATCH = 50
EXPORT_FIGURES = False  # Only enables writing chart PNGs into docs/images
TABLES = dict(DEFAULT_TABLES)
assert DATA_MODE in {"reuse", "create"}
assert isinstance(ACCOUNTS_PER_BATCH, int) and ACCOUNTS_PER_BATCH > 0

## 3 — Read-only Unity Catalog preflight

Checking the complete set of eight destinations before any possible write. Mixed present/missing states require manual investigation.

In [0]:
import re
valid_identifier = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*$")
assert len(TABLES) == 8
assert len(set(TABLES.values())) == 8
assert all(valid_identifier.fullmatch(t) for t in TABLES.values())
assert set(TABLES.values()).isdisjoint(PROTECTED_TABLES | {HISTORY_SILVER, HISTORY_GOLD})

exists_by_role = {role: spark.catalog.tableExists(t) for role, t in TABLES.items()}
for role, name in TABLES.items():
    print(f"{role:20s} {name} exists={exists_by_role[role]}")

if DATA_MODE == "reuse":
    assert all(exists_by_role.values()), "Reuse requires all eight tables; inspect missing destinations"
else:
    assert not any(exists_by_role.values()), "Create requires all eight destinations to be absent; do not overwrite"
print("Preflight PASSED; mode:", DATA_MODE)

## 4 — Generate and verify historical replay (create mode only)

When reusing persisted tables, skip generation entirely. Historical SHA-256 is checked for a fresh creation. To verify that the regenerated development data matches the already-persisted development table, their hashes are compared against eachother. An initial run was previously verified the match to the original 120,000 records.

In [0]:
generated = None
if DATA_MODE == "create":
    generated = prepare_holdout_data(
        spark, n_existing_accounts=N_EXISTING_ACCOUNTS,
        historical_days=HISTORICAL_DAYS, n_future_days=FUTURE_DAYS,
        n_new_accounts=NEW_ACCOUNTS, start_date=START_DATE,
        random_seed=SIMULATION_SEED,
        corruption_seed_temporal=TEMPORAL_CORRUPTION_SEED,
        corruption_seed_account=ACCOUNT_CORRUPTION_SEED,
        history_silver_table=HISTORY_SILVER,
        history_context_days=6, inject_quality_issues=True,
        validate=True, persist=False, tables=TABLES,
    )
    reference = generated["historical_replay"].sort_values(
        ["account_id", "date"]
    ).reset_index(drop=True)
    actual_hash = hashlib.sha256(
        pd.util.hash_pandas_object(reference, index=False).values.tobytes()
    ).hexdigest()
    assert len(reference) == N_EXISTING_ACCOUNTS * HISTORICAL_DAYS
    assert actual_hash == EXPECTED_HISTORY_SHA256, "Historical replay mismatch; refusing persistence"
    print("Original historical SHA-256 verified:", actual_hash)
else:
    print("Reusing persisted tables; no synthetic generation or replay performed.")

## 5 — Create eight tables (create mode only)

**Never rerun this cell after any tables were created.** The writer uses `.create()`, which fails rather than replacing an existing table. Creation is not atomic across all eight tables; a partial failure must be reviewed manually.

In [0]:
if DATA_MODE == "create":
    assert generated is not None and generated["persisted"] is False
    assert all(not spark.catalog.tableExists(t) for t in TABLES.values())
    frames = {
        "temporal_bronze": generated["temporal"].bronze,
        "temporal_silver": generated["temporal"].silver,
        "temporal_gold": generated["temporal"].gold,
        "temporal_truth": generated["temporal"].truth,
        "account_bronze": generated["account"].bronze,
        "account_silver": generated["account"].silver,
        "account_gold": generated["account"].gold,
        "account_truth": generated["account"].truth,
    }
    for role, table in TABLES.items():
        print("Creating:", table)
        frames[role].writeTo(table).using("delta").create()
    print("All eight tables created; existing training tables untouched.")
else:
    print("Reuse mode: no table writes.")

## 6 — Validate the eight persisted tables (read-only)

Checking account counts, date ranges, label isolation, and unique primary keys after cleaning. Bronze intentionally contains duplicate account–dates.

In [0]:
specs = {
    "temporal": {"accounts": 2000, "start": "2026-03-02", "end": "2026-04-30", "rows": 120000},
    "account": {"accounts": 1000, "start": "2026-01-01", "end": "2026-04-30", "rows": 120000},
}
summary_rows = []
for group, spec in specs.items():
    for layer in ("bronze", "silver", "gold", "truth"):
        role = f"{group}_{layer}"
        frame = spark.table(TABLES[role])
        row = frame.agg(
            F.count("*").alias("rows"),
            F.countDistinct("account_id").alias("accounts"),
            F.min(F.to_date("date")).alias("start"),
            F.max(F.to_date("date")).alias("end"),
        ).first().asDict()
        expected_rows = spec["rows"] + spec["rows"] * 5 // 1000 if layer == "bronze" else spec["rows"]
        assert row["rows"] == expected_rows, (role, row)
        assert row["accounts"] == spec["accounts"], (role, row)
        assert str(row["start"]) == spec["start"] and str(row["end"]) == spec["end"], (role, row)
        labels = {"risk_state", "risk_state_name"}
        assert (labels.issubset(frame.columns) if layer == "truth" else labels.isdisjoint(frame.columns)), role
        if layer != "bronze":
            has_duplicates = frame.groupBy("account_id", "date").count().filter(F.col("count") > 1).limit(1).count()
            assert has_duplicates == 0, f"Duplicate processed keys in {role}"
        summary_rows.append({"table": role, **row})

print(pd.DataFrame(summary_rows).to_string(index=False))
assert all(spark.catalog.tableExists(name) for name in TABLES.values())
print("Eight persisted tables VERIFIED. No writes performed in validation.")

## 7 — Build scoring context (read-only)

Temporal prediction windows require six historical Gold timesteps per known account. New accounts already have their complete 120-day Gold trajectories. No truth labels enter scoring context.

In [0]:
historic_gold = spark.table(HISTORY_GOLD).drop("risk_state", "risk_state_name")
temporal_gold = spark.table(TABLES["temporal_gold"])
account_gold = spark.table(TABLES["account_gold"])

context_window = Window.partitionBy("account_id").orderBy(F.col("date").desc())
historic_context = (
    historic_gold.filter(F.to_date("date") < F.lit(FUTURE_START.date()))
    .withColumn("_rank", F.row_number().over(context_window))
    .filter(F.col("_rank") <= 6).drop("_rank")
)
scoring_contexts = {
    "temporal": historic_context.unionByName(temporal_gold, allowMissingColumns=True),
    "account": account_gold,
}

# Evaluating the counts before invoking the model.
assert historic_context.count() == N_EXISTING_ACCOUNTS * 6
assert scoring_contexts["temporal"].count() == 132000
assert scoring_contexts["account"].count() == 120000
for name, frame in scoring_contexts.items():
    assert {"risk_state", "risk_state_name"}.isdisjoint(frame.columns), name
    assert frame.groupBy("account_id", "date").count().filter(F.col("count") > 1).limit(1).count() == 0
print("Scoring context VERIFIED: temporal=132,000, account=120,000")

## 8 — Registered LSTM v1: account-batched inference (opt-in)

Loads pinned model and preprocessing **once per holdout group**, processes 50 accounts at a time, and avoids loading all sequences simultaneously. It still collects prediction dictionaries for evaluation. The cell does not persist predictions. Set `RUN_INFERENCE=True` in Section 2 when ready.

In [0]:
prediction_outputs = {}
inference_summaries = {}

if RUN_INFERENCE:
    import mlflow
    mlflow.set_tracking_uri("databricks")
    mlflow.set_registry_uri("databricks-uc")

    for name in ("temporal", "account"):
        selected_predictions = []
        def capture_batch(batch_id, predictions):
            kept = [
                p for p in predictions
                if name != "temporal" or pd.Timestamp(p["time"]) >= FUTURE_START
            ]
            selected_predictions.extend(kept)
            print(f"{name} / batch {batch_id}: {len(kept):,} evaluation predictions")

        output = run_batched_inference_pipeline(
            scoring_contexts[name],
            config={"model_version": MODEL_VERSION, "boundary_width": 0},
            accounts_per_batch=ACCOUNTS_PER_BATCH,
            on_batch=capture_batch,
        )
        prediction_outputs[name] = selected_predictions
        inference_summaries[name] = {
            "model_name": output["model_name"], "model_version": output["model_version"],
            "n_accounts": output["n_accounts"], "n_batches": output["n_batches"],
            "n_sequences": output["n_sequences"],
            "n_evaluation_predictions": len(selected_predictions),
        }
        print(inference_summaries[name])

    assert len(prediction_outputs["temporal"]) == 120000
    assert len(prediction_outputs["account"]) == 120000
    print("Both holdout groups scored successfully.")
else:
    print("RUN_INFERENCE=False: model scoring skipped. Existing UC tables are unchanged.")

## 9 — Post-development evaluation: Date, Account and Joint Holdouts

| Account identity ↓ / Date → | Historical: Jan 1–Mar 1 | Future: Mar 2–Apr 30 |
|---|---|---|
| **Known accounts (2,000)** | Development dataset (train/validation/test) | **Date Holdout (120k)** |
| **New accounts (1,000)** | **Account Holdout (60k)** | **Joint Holdout (60k)** |

These holdouts were generated *after model development*; they are not the development test split. All are drawn from the same simulator, so results are not external real-bank validation. Predictions join to simulator labels using both `(account_id, date)`. 
Regenerating the `development dataset` can verify whether the generator produces the exact "existing development data that were already created before model development" (Step 4 above).

In [0]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

COHORT_LABELS = {"date": "Date Holdout", "account": "Account Holdout", "joint": "Joint Holdout"}
CLASSES = [0, 1, 2, 3]
CLASS_NAMES = ["normal", "elevated_risk", "liquidity_stress", "delinquency_risk"]
cohorts = {}
metrics_rows = []
per_class_reports = {}
confusion_matrices = {}

if RUN_INFERENCE:
    def align(predictions, truth_table):
        pred = pd.DataFrame([
            {"account_id": p["group"], "date": pd.Timestamp(p["time"]),
             "y_pred": int(p["predicted_class"])} for p in predictions
        ])
        truth = spark.table(truth_table).select("account_id", "date", "risk_state").toPandas()
        truth["date"] = pd.to_datetime(truth["date"])
        truth = truth.rename(columns={"risk_state": "y_true"})
        assert not pred.empty
        assert not pred.duplicated(["account_id", "date"]).any()
        assert not truth.duplicated(["account_id", "date"]).any()
        joined = pred.merge(truth, how="outer", on=["account_id", "date"],
                            validate="one_to_one", indicator=True)
        assert joined["_merge"].eq("both").all(), joined["_merge"].value_counts().to_dict()
        assert joined["y_pred"].isin(CLASSES).all() and joined["y_true"].isin(CLASSES).all()
        return joined.drop(columns="_merge")

    date_all = align(prediction_outputs["temporal"], TABLES["temporal_truth"])
    account_all = align(prediction_outputs["account"], TABLES["account_truth"])
    cohorts = {
        "date": date_all,
        "account": account_all.loc[account_all["date"] < FUTURE_START].copy(),
        "joint": account_all.loc[account_all["date"] >= FUTURE_START].copy(),
    }
    for name, rows in cohorts.items():
        expected_n = 120000 if name == "date" else 60000
        assert len(rows) == expected_n, (name, len(rows))
        y_true = rows["y_true"].to_numpy(dtype=int)
        y_pred = rows["y_pred"].to_numpy(dtype=int)
        metrics_rows.append({
            "cohort": name, "display_name": COHORT_LABELS[name], "n_observations": len(rows),
            "n_accounts": rows["account_id"].nunique(),
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, labels=CLASSES,
                                  average="macro", zero_division=0),
        })
        per_class_reports[name] = classification_report(
            y_true, y_pred, labels=CLASSES, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0,
        )
        confusion_matrices[name] = confusion_matrix(y_true, y_pred, labels=CLASSES)

    evaluation_summary = pd.DataFrame(metrics_rows)
    print(evaluation_summary.drop(columns="cohort").round(4).to_string(index=False))
    for name in ("date", "account", "joint"):
        print("\n", COHORT_LABELS[name], "classification report")
        print(pd.DataFrame(per_class_reports[name]).T.round(4).to_string())
        print("Confusion matrix (true rows, predicted columns):\n", confusion_matrices[name])
else:
    print("RUN_INFERENCE=False: evaluation skipped; turn it on to recompute metrics.")

## 10 — Evaluation charts and optional figure export

Charts are computed from the current evaluation results, never hardcoded. Saving images into `docs/images` is optional; review and commit them during the documentation stage.

In [0]:
import matplotlib.pyplot as plt

if RUN_INFERENCE:
    plot_data = evaluation_summary.set_index("cohort").loc[["date", "account", "joint"]]
    plot_data.index = [COHORT_LABELS[key] for key in plot_data.index]
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_data[["accuracy", "macro_f1"]].plot.bar(ax=ax, rot=0)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score")
    ax.set_title("Registered LSTM v1 — post-development holdout scores")
    ax.legend(loc="lower right")
    fig.tight_layout()
    if EXPORT_FIGURES:
        image_dir = project_root / "docs" / "images"
        image_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(image_dir / "holdout_overall_metrics.png", dpi=180, bbox_inches="tight")
    plt.show()

    per_class_f1 = pd.DataFrame({
        group: [per_class_reports[group][label]["f1-score"] for label in CLASS_NAMES]
        for group in ("date", "account", "joint")
    }, index=CLASS_NAMES)
    per_class_f1 = per_class_f1.rename(columns=COHORT_LABELS)
    fig, ax = plt.subplots(figsize=(9, 5))
    per_class_f1.plot.bar(ax=ax, rot=15)
    ax.set_ylim(0, 1)
    ax.set_ylabel("F1")
    ax.set_title("Class-wise F1 by post-development holdout")
    fig.tight_layout()
    if EXPORT_FIGURES:
        fig.savefig(image_dir / "holdout_classwise_f1.png", dpi=180, bbox_inches="tight")
    plt.show()

    for group, matrix in confusion_matrices.items():
        fig, ax = plt.subplots(figsize=(7, 5))
        im = ax.imshow(matrix, cmap="Blues")
        ax.set_xticks(range(4), labels=CLASS_NAMES, rotation=35, ha="right")
        ax.set_yticks(range(4), labels=CLASS_NAMES)
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title(f"{COHORT_LABELS[group]} — confusion matrix")
        for (i, j), value in np.ndenumerate(matrix):
            ax.text(j, i, f"{value:,}", ha="center", va="center",
                    color="white" if value > matrix.max() / 2 else "black", fontsize=9)
        fig.colorbar(im, ax=ax)
        fig.tight_layout()
        if EXPORT_FIGURES:
            fig.savefig(image_dir / f"confusion_matrix_{group}.png", dpi=180, bbox_inches="tight")
        plt.show()
else:
    print("No charts generated because inference/evaluation was skipped.")